# Knowledge Graph Question Answering (KGQA) over Wikidata

**Advanced NLP Course — Exercise Session**  
Leibniz Universität Hannover

This notebook shows how to query structured knowledge directly by 1) linking the mentioned entities to a KB ID, 2) parsing the question into a SPARQL query, 3) executing the SPARQL query against Wikidata, and 4) verbalizing the results.

In [1]:
import requests

WIKIDATA_SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
WIKIDATA_API_ENDPOINT = "https://www.wikidata.org/w/api.php"
USER_AGENT = "AdvancedNLP/1.0"

# 1 Entity linking

Given a mention like `"Albert Einstein"`, use Wikidata's `wbsearchentities` API action to find the best-matching QID (e.g. `Q937`).

In [2]:
def link_entity(mention, language="en"):
    '''Return (qid, label) for the top Wikidata search match of `mention`, or (None, None).'''
    params = {
        "action": "wbsearchentities",
        "search": mention,
        "language": language,
        "format": "json",
        "limit": 1,
    }
    resp = requests.get(WIKIDATA_API_ENDPOINT, params=params,
                         headers={"User-Agent": USER_AGENT}, timeout=10)
    data = resp.json()
    results = data.get("search", [])
    if not results:
        return None, None
    return results[0]["id"], results[0].get("label", mention)

# quick test
print(link_entity("Albert Einstein"))
print(link_entity("Bayern Munich"))


('Q937', 'Albert Einstein')
('Q15789', 'FC Bayern Munich')


# 2. Running a SPARQL query

Implementing a helper that sends a SPARQL query string to the Wikidata Query Service and returns the JSON bindings.

In [4]:
import requests

def run_sparql(query):
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/sparql-results+json",
    }
    response = requests.get(
        WIKIDATA_SPARQL_ENDPOINT,
        params={"query": query},
        headers=headers,
        timeout=30,
    )

    response.raise_for_status()
    return response.json()["results"]["bindings"]

query = """
SELECT ?placeLabel WHERE {
  wd:Q937 wdt:P19 ?place .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

print(run_sparql(query))

[{'placeLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Ulm'}}]


# 3. Template-based semantic parser


In [5]:
import re


TEMPLATES = [
    {
        "pattern": re.compile(r"where was (.+?)[?]?$", re.I),
        "property": "P19",
        "direction": "forward",
    },
    {
        "pattern": re.compile(r"what is the capital of (.+?)[?]?$", re.I),
        "property": "P36",
        "direction": "forward",
    },
    {
        "pattern": re.compile(r"who are the football players (?:in|of) (.+?)[?]?$", re.I),
        "property": "P54",
        "direction": "backward",
    }
]

def parse_question(question):
    '''Match `question` against TEMPLATES, link the extracted entity mention,
    and return a SPARQL query string, or None if no template matches / linking fails.'''
    for template in TEMPLATES:
        m = template["pattern"].search(question)
        if not m:
            continue
        mention = m.group(1).strip()
        qid, label = link_entity(mention)

        if qid is None:
            return None
        prop = template["property"]
        if template["direction"] == "forward":
            triple = f"wd:{qid} wdt:{prop} ?answer ."
        else:
            triple = f"?answer wdt:{prop} wd:{qid} ."
        query = f'''
        SELECT DISTINCT ?answerLabel WHERE {{
          {triple}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        LIMIT 10
        '''
        
        return query
    return None


# 4. Answering a question


In [6]:
def answer_question(question):
    query = parse_question(question)
    if query is None:
        return "Sorry, I don't have a template for this question."
    bindings = run_sparql(query)
    if not bindings:
        return "No answer found in Wikidata for this query."
    answers = [b["answerLabel"]["value"] for b in bindings]
    return ", ".join(sorted(set(answers)))

In [7]:
for q in [
    "Where was Albert Einstein born?",
    "What is the capital of Germany?",
    "Who are the football players in Bayern Munich?",
]:
    print(f"Q: {q}\nA: {answer_question(q)}\n")

Q: Where was Albert Einstein born?
A: Sorry, I don't have a template for this question.

Q: What is the capital of Germany?
A: Berlin

Q: Who are the football players in Bayern Munich?
A: Jan Wouters, Jorginho, Karl Adam, Mark Hughes, Markus Münch, Max Grün, Nils Petersen, Patrik Andersson, Stefan Reuter, Thomas Linke

